# Industry Classification & Fill Strategy

## Objective

This notebook implements and evaluates the strategy used to recover missing
`industry` values in the Mexico P&C portfolio.

The objective is to maximize recovery while maintaining transparency and
classification reliability.

In [5]:
import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/portfolio_mexico.csv")

df = pd.read_csv(DATA_PATH)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Saving portfolio_mexico.csv to portfolio_mexico (1).csv
Rows: 50,441
Columns: 11


In [6]:
df["industry_original"] = df["industry"]

df["industry_final"] = df["industry"]

df["fill_method"] = np.where(
    df["industry"].notna(),
    "original",
    "unresolved"
)

df["fill_confidence"] = np.where(
    df["industry"].notna(),
    1.0,
    np.nan
)

In [7]:
known_industry = df.dropna(subset=["industry"])

client_industry_map = (
    known_industry
    .groupby("ClientId")["industry"]
    .first()
)

In [8]:
recoverable_mask = (
    df["industry"].isna()
    & df["ClientId"].isin(client_industry_map.index)
)

print(f"Records recoverable from client history: {recoverable_mask.sum():,}")

Records recoverable from client history: 16,159


In [9]:
df.loc[
    recoverable_mask,
    "industry_final"
] = df.loc[
    recoverable_mask,
    "ClientId"
].map(client_industry_map)

df.loc[
    recoverable_mask,
    "fill_method"
] = "same_client_history"

df.loc[
    recoverable_mask,
    "fill_confidence"
] = 1.0

In [10]:
df["fill_method"].value_counts()

,count
fill_method,
original,32810
same_client_history,16159
unresolved,1472


In [11]:
features = [
    "state",
    "municipality",
    "coverage_type",
    "premium",
    "sum_insured"
]

target = "industry"

In [12]:
from sklearn.model_selection import GroupShuffleSplit

In [13]:
labeled = df[df["industry"].notna()].copy()

X = labeled[features]
y = labeled[target]

groups = labeled["ClientId"]

In [14]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [15]:
train_clients = set(
    labeled.iloc[train_idx]["ClientId"]
)

test_clients = set(
    labeled.iloc[test_idx]["ClientId"]
)

print(
    "Clients appearing in both sets:",
    len(train_clients.intersection(test_clients))
)

Clients appearing in both sets: 0


##Baseline

In [16]:
from sklearn.metrics import accuracy_score, f1_score

majority_class = y_train.mode()[0]

baseline_predictions = np.full(
    len(y_test),
    majority_class
)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

baseline_macro_f1 = f1_score(
    y_test,
    baseline_predictions,
    average="macro"
)

print(f"Majority class: {majority_class}")
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")
print(f"Baseline Macro F1: {baseline_macro_f1:.4f}")

Majority class: Construcción
Baseline Accuracy: 0.1008
Baseline Macro F1: 0.0122


##Logistic Regression

In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

In [18]:
categorical_features = [
    "state",
    "municipality",
    "coverage_type"
]

numeric_features = [
    "premium",
    "sum_insured"
]

In [19]:
preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    (
        "numeric",
        StandardScaler(),
        numeric_features
    )
])

In [20]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        LogisticRegression(
            max_iter=1500,
            class_weight="balanced"
        )
    )
])

In [21]:
logistic_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['state', 'municipality',
                                                   'coverage_type']),
                                                 ('numeric', StandardScaler(),
                                                  ['premium',
                                                   'sum_insured'])])),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=1500))])

In [22]:
logistic_predictions = logistic_model.predict(X_test)

logistic_accuracy = accuracy_score(
    y_test,
    logistic_predictions
)

logistic_macro_f1 = f1_score(
    y_test,
    logistic_predictions,
    average="macro"
)

print(
    f"Logistic Regression Accuracy: "
    f"{logistic_accuracy:.4f}"
)

print(
    f"Logistic Regression Macro F1: "
    f"{logistic_macro_f1:.4f}"
)

Logistic Regression Accuracy: 0.0509
Logistic Regression Macro F1: 0.0488


##Random Forest

In [23]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        )
    )
])

In [24]:
random_forest_model.fit(
    X_train,
    y_train
)

rf_predictions = random_forest_model.predict(
    X_test
)

In [25]:
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_macro_f1 = f1_score(
    y_test,
    rf_predictions,
    average="macro"
)

print(
    f"Random Forest Accuracy: "
    f"{rf_accuracy:.4f}"
)

print(
    f"Random Forest Macro F1: "
    f"{rf_macro_f1:.4f}"
)

Random Forest Accuracy: 0.0610
Random Forest Macro F1: 0.0581


## Model Evaluation Conclusion
The evaluation concluded that the supervised learning models lack the ability to generalize industry classification for new clients.

To ensure a realistic validation, the evaluation was designed to prevent policies from the same client from being split between the training and test sets. Under these conditions, it became clear that variables such as geography, coverage type, premiums, and sums insured do not provide a strong enough predictive signal.

Consequently, automated machine learning predictions will not be used to fill in the missing industry data.

##Semantic signal

In [26]:
unresolved = df[
    df["industry_final"].isna()
].copy()

print(f"Unresolved policies: {len(unresolved):,}")
print(f"Unresolved clients: {unresolved['ClientId'].nunique():,}")

unresolved[
    ["ClientId", "client_name", "coverage_type"]
].head(20)

Unresolved policies: 1,472
Unresolved clients: 1,171


,ClientId,client_name,coverage_type
18,PSVG549915I9B,Desarrollos Morelos S.C.,Transporte de Mercancías
72,QZBG981025PGH,Inversiones Regiomontana S.A.P.I. de C.V.,Flotilla Vehicular
80,PGOB132625RFM,Constructora Capitalina S.C.,Responsabilidad Civil
108,FSCO072278Y17,Corporativo Central S. de R.L. de C.V.,Flotilla Vehicular
199,VNJL283097K4M,Agroindustrias Prime S.A.,Responsabilidad Civil
206,UCXI155392MRE,Desarrollos Beta S.A.P.I. de C.V.,Agrícola
210,ZHFS0646098P8,Inversiones Atlántico S.A.P.I. de C.V.,Ingeniería
248,UWGE641015S5O,Agroindustrias Regiomontana S.A. de C.V.,Ingeniería
294,QBOL087146BGD,Grupo Guerrero S.C.,Incendio y Terremoto
311,HYSH7261280YW,Jabalí S.A. de C.V.,Misceláneos


In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit

In [28]:
labeled = df[
    df["industry_original"].notna()
].copy()

X = labeled["client_name"]
y = labeled["industry_original"]
groups = labeled["ClientId"]

In [29]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [30]:
text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    )
])

In [31]:
text_model.fit(
    X_train,
    y_train
)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(min_df=2, ngram_range=(1, 2),
                                 strip_accents='unicode')),
                ('classifier',
                 LogisticRegression(class_weight='balanced', max_iter=2000))])

In [32]:
text_predictions = text_model.predict(
    X_test
)

text_accuracy = accuracy_score(
    y_test,
    text_predictions
)

text_macro_f1 = f1_score(
    y_test,
    text_predictions,
    average="macro"
)

print(
    f"Client Name Accuracy: "
    f"{text_accuracy:.4f}"
)

print(
    f"Client Name Macro F1: "
    f"{text_macro_f1:.4f}"
)

Client Name Accuracy: 0.0665
Client Name Macro F1: 0.0623


In [33]:
text_probabilities = text_model.predict_proba(
    X_test
)

max_probabilities = text_probabilities.max(
    axis=1
)

In [34]:
evaluation = pd.DataFrame({
    "actual": y_test.values,
    "predicted": text_predictions,
    "confidence": max_probabilities
})

evaluation["correct"] = (
    evaluation["actual"]
    == evaluation["predicted"]
)

evaluation.head()

,actual,predicted,confidence,correct
0,Transporte y Logística,Transporte y Logística,0.330110,True
1,Hotelería y Turismo,Tecnología,0.222801,False
2,Manufactura,Servicios Financieros,0.357389,False
3,Educación,Agricultura,0.561787,False
4,Manufactura,Hotelería y Turismo,0.391453,False


In [35]:
thresholds = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.95
]

results = []

for threshold in thresholds:

    subset = evaluation[
        evaluation["confidence"] >= threshold
    ]

    if len(subset) == 0:
        continue

    accuracy = subset["correct"].mean()

    coverage = (
        len(subset)
        / len(evaluation)
    )

    results.append({
        "threshold": threshold,
        "accuracy": accuracy,
        "coverage": coverage,
        "records": len(subset)
    })

threshold_analysis = pd.DataFrame(results)

threshold_analysis

,threshold,accuracy,coverage,records
0,0.50,0.038911,0.100567,514
1,0.60,0.016760,0.035023,179
2,0.70,0.120000,0.004891,25
3,0.80,0.000000,0.001761,9
4,0.90,0.000000,0.001761,9
5,0.95,0.000000,0.001565,8


In [36]:
print(text_accuracy)
print(text_macro_f1)

threshold_analysis

0.06652318528663667
0.06233609145815373


,threshold,accuracy,coverage,records
0,0.50,0.038911,0.100567,514
1,0.60,0.016760,0.035023,179
2,0.70,0.120000,0.004891,25
3,0.80,0.000000,0.001761,9
4,0.90,0.000000,0.001761,9
5,0.95,0.000000,0.001565,8


### Client Name Model Assessment

Client names were evaluated as an additional semantic signal using TF-IDF
features and Logistic Regression.

To simulate the actual unresolved population, validation was performed at the
client level, ensuring that clients in the test set were not present during
training.

The model achieved:

- Accuracy: 6.65%
- Macro F1: 6.23%

Increasing the prediction probability threshold did not improve reliability.
In fact, the small number of predictions with very high model probabilities
were not consistently correct.

Therefore, client-name predictions are not considered sufficiently reliable
for automatic industry assignment.

No missing industry values were filled using this model.

In [40]:
import re
import unicodedata

def normalize_text(text):

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text

In [41]:
df["client_name_normalized"] = (
    df["client_name"]
    .apply(normalize_text)
)

In [42]:
df[
    ["client_name", "client_name_normalized"]
].head(10)

,client_name,client_name_normalized
0,Productos Veracruzana S.C.,productos veracruzana s c
1,Tecnologías Bajío S.A. de C.V.,tecnologias bajio s a de c v
2,Tecnologías Bajío S.A. de C.V.,tecnologias bajio s a de c v
3,Inmobiliaria Torres S.A.,inmobiliaria torres s a
4,Transportes Mendoza S.A. de C.V.,transportes mendoza s a de c v
5,Tecnologias Bajio S.A. de C.V.,tecnologias bajio s a de c v
6,Inversiones Atlántico S.A. de C.V.,inversiones atlantico s a de c v
7,Inversiones Morelos S.C.,inversiones morelos s c
8,Grupo Beta S.A.,grupo beta s a
9,Tecnologías Torres S.C.,tecnologias torres s c


In [43]:
client_labels = (
    df[
        df["industry_original"].notna()
    ][
        [
            "ClientId",
            "client_name_normalized",
            "industry_original"
        ]
    ]
    .drop_duplicates()
)

In [44]:
print(
    "Labeled policies:",
    df["industry_original"].notna().sum()
)

print(
    "Unique labeled clients:",
    client_labels["ClientId"].nunique()
)

print(
    "Unique client-industry-name combinations:",
    len(client_labels)
)

Labeled policies: 32810
Unique labeled clients: 7394
Unique client-industry-name combinations: 8477


In [45]:
client_words = (
    client_labels
    .assign(
        word=lambda x:
        x["client_name_normalized"].str.split()
    )
    .explode("word")
)

client_words = client_words[
    client_words["word"].str.len() >= 4
]

In [46]:
client_word_industry = (
    client_words
    .groupby(
        ["word", "industry_original"]
    )["ClientId"]
    .nunique()
    .reset_index(name="clients")
)

In [47]:
client_word_totals = (
    client_word_industry
    .groupby("word")["clients"]
    .sum()
    .rename("total_clients")
    .reset_index()
)

client_word_stats = (
    client_word_industry
    .merge(
        client_word_totals,
        on="word"
    )
)

client_word_stats["precision"] = (
    client_word_stats["clients"]
    / client_word_stats["total_clients"]
)

In [48]:
candidate_rules_client = (
    client_word_stats[
        (client_word_stats["total_clients"] >= 10)
        & (client_word_stats["precision"] >= 0.90)
    ]
    .sort_values(
        ["precision", "total_clients"],
        ascending=[False, False]
    )
)

candidate_rules_client.head(50)

,word,industry_original,clients,total_clients,precision


In [49]:
keywords = [
    "inmobiliaria",
    "constructora",
    "tecnologias",
    "agroindustrias",
    "distribuidora",
    "industrias",
    "productos",
    "servicios",
    "desarrollos"
]

keyword_results = []

for keyword in keywords:

    subset = client_labels[
        client_labels["client_name_normalized"]
        .str.contains(
            rf"\b{keyword}\b",
            regex=True,
            na=False
        )
    ]

    if len(subset) == 0:
        continue

    counts = (
        subset["industry_original"]
        .value_counts()
    )

    dominant_industry = counts.index[0]
    dominant_count = counts.iloc[0]

    keyword_results.append({
        "keyword": keyword,
        "clients": subset["ClientId"].nunique(),
        "dominant_industry": dominant_industry,
        "dominant_count": dominant_count,
        "purity": dominant_count / len(subset)
    })

keyword_validation = pd.DataFrame(
    keyword_results
)

keyword_validation

,keyword,clients,dominant_industry,dominant_count,purity
0,inmobiliaria,477,Construcción,71,0.140594
1,constructora,439,Manufactura,66,0.142857
2,tecnologias,475,Manufactura,69,0.137177
3,agroindustrias,481,Manufactura,85,0.168651
4,distribuidora,511,Manufactura,82,0.151013
5,industrias,483,Manufactura,97,0.185824
6,productos,511,Manufactura,76,0.140481
7,servicios,462,Manufactura,69,0.140244
8,desarrollos,491,Manufactura,80,0.153846


In [37]:
keywords = [
    "inmobiliaria",
    "constructora",
    "tecnologias",
    "agroindustrias",
    "distribuidora",
    "industrias",
    "productos",
    "servicios",
    "desarrollos"
]

keyword_results = []

for keyword in keywords:

    subset = client_labels[
        client_labels["client_name_normalized"]
        .str.contains(
            rf"\b{keyword}\b",
            regex=True,
            na=False
        )
    ]

    if len(subset) == 0:
        continue

    counts = (
        subset["industry_original"]
        .value_counts()
    )

    dominant_industry = counts.index[0]
    dominant_count = counts.iloc[0]

    keyword_results.append({
        "keyword": keyword,
        "clients": subset["ClientId"].nunique(),
        "dominant_industry": dominant_industry,
        "dominant_count": dominant_count,
        "purity": dominant_count / len(subset)
    })

keyword_validation = pd.DataFrame(
    keyword_results
)

keyword_validation

NameError: name 'client_labels' is not defined

### External Enrichment Feasibility Assessment

External enrichment was considered for the remaining unresolved clients.

Before querying external sources or a language model, the semantic relationship
between client names and known industry labels was evaluated.

Terms that would normally appear strongly indicative of an economic activity,
such as `inmobiliaria`, `constructora`, `tecnologias`, and `agroindustrias`,
showed very low industry purity at the client level.

For example:

- `inmobiliaria`: 14.1% dominant-industry purity
- `constructora`: 14.3%
- `tecnologias`: 13.7%

This indicates that client names in the dataset appear to be synthetic,
anonymized, or otherwise weakly related to the actual industry label.

Consequently, linking these names to real-world companies through web searches
or a language model could introduce incorrect external information.

External enrichment was therefore not used for automatic classification.

Residual cases are explicitly classified as `Unknown`, prioritizing data
integrity and transparency over an artificial 100% fill rate.

In [50]:
unresolved_mask = df["industry_final"].isna()

df.loc[
    unresolved_mask,
    "industry_final"
] = "Unknown"

df.loc[
    unresolved_mask,
    "fill_method"
] = "unresolved"

df.loc[
    unresolved_mask,
    "fill_confidence"
] = 0.0

In [51]:
df["fill_method"].value_counts()

,count
fill_method,
original,32810
same_client_history,16159
unresolved,1472


In [52]:
print(
    "Remaining missing industry_final:",
    df["industry_final"].isna().sum()
)

Remaining missing industry_final: 0


##Fill Quality Report

In [53]:
original_missing = df["industry_original"].isna().sum()

recovered_client_history = (
    df["fill_method"] == "same_client_history"
).sum()

unresolved_count = (
    df["fill_method"] == "unresolved"
).sum()

recovery_rate = (
    recovered_client_history
    / original_missing
)

portfolio_classified_rate = (
    df["industry_final"] != "Unknown"
).mean()

print(f"Original missing: {original_missing:,}")
print(
    f"Recovered from client history: "
    f"{recovered_client_history:,}"
)
print(
    f"Residual Unknown: "
    f"{unresolved_count:,}"
)

print(
    f"Missing-value recovery rate: "
    f"{recovery_rate:.2%}"
)

print(
    f"Final classified portfolio: "
    f"{portfolio_classified_rate:.2%}"
)

Original missing: 17,631
Recovered from client history: 16,159
Residual Unknown: 1,472
Missing-value recovery rate: 91.65%
Final classified portfolio: 97.08%


In [54]:
df = df.drop(
    columns=["client_name_normalized"],
    errors="ignore"
)

In [55]:
df.to_csv(
    "portfolio_mexico_filled.csv",
    index=False
)

In [56]:
from google.colab import files

files.download(
    "portfolio_mexico_filled.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
assert len(df) == 50441

assert df["policy_id"].nunique() == 50441

assert df["industry_final"].isna().sum() == 0

assert (
    df.loc[
        df["industry_original"].notna(),
        "industry_original"
    ]
    ==
    df.loc[
        df["industry_original"].notna(),
        "industry_final"
    ]
).all()

print("All validation checks passed.")

All validation checks passed.
